# Evaluate similarity suggestions

In [1]:
%load_ext autoreload

In [2]:
import gc
import os
import pickle
import warnings
from os.path import join
from IPython.display import display, Markdown, display_html

import scanpy as sc
import pandas as pdx 
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt

In [3]:
%autoreload
from datasim.dataset_ot import DatasetMapping

## Load and preprocess data

In [4]:
DATA_PATH = "/vol/data/dataset-similarity/preprocessed"
CACHE_DIR = "/vol/data/dataset-similarity/cache"
FIG_DIR = "/vol/data/dataset-similarity/figures"


QUERY_DATASET = "7d7cabfd-1d1f-40af-96b7-26a0825a306d"
REF_DATASET = "ced320a1-29f3-47c1-a735-513c7084d508"
N_TOP_GENES = 1500

In [5]:
cache_file = join(
    CACHE_DIR, 
    "+".join([QUERY_DATASET, REF_DATASET, f"{N_TOP_GENES}HVGs"]) + ".pickle"
)
if os.path.isfile(cache_file):
    print("Using cached files...")
    with open(cache_file, "rb") as f:
        adata_query, adata_ref = pickle.load(f)
else:
    # load and preprocess data
    adata_query, adata_ref = DatasetMapping.preprocess_adatas(
        sc.read_h5ad(join(DATA_PATH, f"{QUERY_DATASET}.h5ad")),
        sc.read_h5ad(join(DATA_PATH, f"{REF_DATASET}.h5ad")),
        n_top_genes=N_TOP_GENES
    )
    with open(cache_file, "wb") as f:
        pickle.dump((adata_query, adata_ref), f)

adata_query.obs["cell_type_author"] = adata_query.obs["ct2"]
gc.collect();

Using cached files...


In [6]:
# normalize data for DE tests
sc.pp.normalize_total(adata_query, target_sum=1e4)
sc.pp.normalize_total(adata_ref, target_sum=1e4)
sc.pp.log1p(adata_query)
sc.pp.log1p(adata_ref)

In [7]:
# Use human readable gene names for evaluation
adata_query.var.set_index("feature_name", inplace=True)
adata_ref.var.set_index("feature_name", inplace=True)

In [8]:
adata_query

AnnData object with n_obs × n_vars = 600929 × 1500
    obs: 'assay', 'cell_type', 'development_stage', 'disease', 'donor_id', 'is_primary_data', 'sex', 'suspension_type', 'tissue', 'ct1', 'ct2', 'ct3', 'cell_type_author'
    uns: 'log1p'

In [9]:
adata_ref

AnnData object with n_obs × n_vars = 1058909 × 1500
    obs: 'assay', 'cell_type', 'development_stage', 'disease', 'donor_id', 'is_primary_data', 'sex', 'suspension_type', 'tissue', 'author_cell_type', 'cell_type_author'
    uns: 'log1p'

In [10]:
MODEL_VERSION = "_tau=1.00"

cluster_mapping = pd.read_parquet(f"/vol/data/dataset-similarity/model-output/cluster_mapping{MODEL_VERSION}.parquet")
cluster_distance = pd.read_parquet(f"/vol/data/dataset-similarity/model-output/cluster_distance{MODEL_VERSION}.parquet")

In [11]:
def extract_ontology_mapping(adata):
    return (
        adata.obs[["cell_type_author", "cell_type"]]
        .drop_duplicates()
        .set_index("cell_type_author")["cell_type"]
        .to_dict()
    )


ontology_mapping_query = extract_ontology_mapping(adata_query)
ontology_mapping_ref = extract_ontology_mapping(adata_ref)

## Select most similar clusters

In [12]:
top_n_labels = DatasetMapping.select_most_similar_clusters(
    cluster_mapping, 
    cluster_distance, 
    threshold_mass=0.2,
    threshold_distance=1.1, 
    n_top=None
)
top_n_labels

{'B_Mem': ['IGHMhi_memory_B'],
 'B_Mem_Prolif': ['IGHMlo_memory_B'],
 'B_Naive': ['naive_B'],
 'B_Preplasma': ['atypical_B'],
 'NKT': ['NK', 'CD8+_T_GZMB+'],
 'NK_CD16+': ['CD16+_NK'],
 'NK_CD56++': ['CD56+_NK'],
 'NK_Prolif': [],
 'PB': ['Plasma_B'],
 'PB_Prolif': [],
 'Progen_CLP': [],
 'Progen_CMP': [],
 'Progen_MEP': [],
 'Progen_MPP': [],
 'T4_Mem': ['CD4+_T_cm'],
 'T4_Mem_Prolif': ['CD4+_T', 'CD4+_T_em'],
 'T4_Naive': ['CD4+_T_naive'],
 'T4_Treg': ['Treg'],
 'T8_MAIT': ['MAIT'],
 'T8_Mem': ['CD4+_T_cyt', 'CD8+_T_GZMB+'],
 'T8_Mem_Prolif': ['CD8+_T_GZMK+'],
 'T8_Naive': ['CD8+_T_naive'],
 'T_NK_Prolif': [],
 'Tgd_1': ['CD8+_T_GZMB+'],
 'Tgd_2': ['gdT'],
 'cDC_1': ['cDC1'],
 'cDC_2': ['cDC2', 'cDC'],
 'cM': ['CD14+_Monocyte'],
 'ncM': ['CD16+_Monocyte'],
 'pDC': ['pDC']}

#### Author provided cluster labels

In [13]:
for i, (k, v) in enumerate(top_n_labels.items()):
    display(Markdown(f"*{i+1}*: **{k}**: {v}"))

*1*: **B_Mem**: ['IGHMhi_memory_B']

*2*: **B_Mem_Prolif**: ['IGHMlo_memory_B']

*3*: **B_Naive**: ['naive_B']

*4*: **B_Preplasma**: ['atypical_B']

*5*: **NKT**: ['NK', 'CD8+_T_GZMB+']

...

*26*: **cDC_1**: ['cDC1']

*27*: **cDC_2**: ['cDC2', 'cDC']

*28*: **cM**: ['CD14+_Monocyte']

*29*: **ncM**: ['CD16+_Monocyte']

*30*: **pDC**: ['pDC']

#### Ontology mapped cluster labels

In [14]:
for i, (k, v) in enumerate(top_n_labels.items()):
    display(Markdown(f"*{i+1}*: **{ontology_mapping_query[k]}**: {[ontology_mapping_ref[elem] for elem in v]}"))

*1*: **B cell**: ['memory B cell']

*2*: **B cell**: ['memory B cell']

*3*: **B cell**: ['naive B cell']

*4*: **B cell**: ['mature B cell']

*5*: **natural killer cell**: ['natural killer cell', 'CD8-positive, alpha-beta cytotoxic T cell']

...

*26*: **conventional dendritic cell**: ['CD141-positive myeloid dendritic cell']

*27*: **conventional dendritic cell**: ['CD1c-positive myeloid dendritic cell', 'conventional dendritic cell']

*28*: **classical monocyte**: ['CD14-positive monocyte']

*29*: **non-classical monocyte**: ['CD14-low, CD16-positive monocyte']

*30*: **plasmacytoid dendritic cell**: ['plasmacytoid dendritic cell']

## Evaluate cluster similarity

In [15]:
%autoreload
from datasim.utils import get_differentially_expressed_genes

In [16]:
def highly_expressed_genes(adata, n_genes):
    highly_expressed = {}
    
    for cluster in adata.obs["cell_type_author"].unique():
        avg_expression = np.array(
            adata[adata.obs["cell_type_author"] == cluster].X.mean(axis=0)
        ).flatten()
        highly_expressed_genes_idxs = np.argsort(-avg_expression)[:n_genes]
        highly_expressed[cluster] = {
            "gene": adata.var.index[highly_expressed_genes_idxs].tolist(),
            "average_expression": avg_expression[highly_expressed_genes_idxs]
        }

    return highly_expressed


In [17]:
METHOD = "wilcoxon"
P_VAL_THRESHOLD = 0.01

# ignore warnings here as scanpy.tl.rank_genes_groups throws a lot of warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    de_genes_query = get_differentially_expressed_genes(
        adata_query, 
        "cell_type_author", 
        n_genes=15,
        method=METHOD,
        p_value_threshold=P_VAL_THRESHOLD
    )
    de_genes_ref = get_differentially_expressed_genes(
        adata_ref, 
        "cell_type_author", 
        n_genes=15,
        method=METHOD,
        p_value_threshold=P_VAL_THRESHOLD
    )


In [18]:
highly_expressed_query = highly_expressed_genes(adata_query, n_genes=100)
highly_expressed_ref = highly_expressed_genes(adata_ref, n_genes=100)

In [19]:
def to_hex(m, val):
    rgba = m.to_rgba(val)
    r, g, b, _ = rgba
    return "#{:02x}{:02x}{:02x}".format(int(r*255), int(g*255), int(b*255))


def style_map_val(v, props=""):
    m = cm.ScalarMappable(
        norm=mpl.colors.Normalize(vmin=0.0, vmax=1.0), 
        cmap=plt.get_cmap("Greens")
    )
    return f"background:{to_hex(m, v)};"


def style_dist_val(v, props=""):
    m = cm.ScalarMappable(
        norm=mpl.colors.Normalize(vmin=0.0, vmax=2.0), 
        cmap=plt.get_cmap("RdYlGn_r")
    )
    return f"background:{to_hex(m, v)};"


In [20]:
N_GENES_TO_SHOW = 15


for k, v in top_n_labels.items():
    mapping = cluster_mapping.loc[k]
    distance = cluster_distance.loc[k]
    html = [f"<h1> <b>{k}</b>: </h1>"]

    if v:
        # Add summary for suggestions
        html.append("<b><i>Suggestions:</i></b> <br />")
        ct_name, map_vals, dist_vals = [], [], []
        for s in v:
            ct_name.append(s)
            map_vals.append(mapping[s])
            dist_vals.append(distance[s])
        html.append(
            pd.DataFrame({"OT mass": map_vals, "distance": dist_vals}, index=ct_name)
            .style
            .format("{:.2f}")
            .map(style_map_val, subset=["OT mass"])
            .map(style_dist_val, subset=["distance"])
            .set_table_attributes("style='display:inline'")
            ._repr_html_()
        )
        html.append("<br /><br />")
        # Add summary for differentially expressed genes
        html.append("<b><i>Overlap of DE genes:</i></b> <br /><br />")

        def style_overlap(v, props=""):
            top_n_query_genes = de_genes_query[k]["gene"][:N_GENES_TO_SHOW].tolist()
            return "color:green;" if v in top_n_query_genes else "color:red;"

        html += [
            de_genes_query[k]
            .head(N_GENES_TO_SHOW)
            .copy()
            .style
            .format("{:.2f}", subset=["logfoldchg", "score"])
            .format("{:.3f}", subset=["pval_adj"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"QUERY - {k}")
            ._repr_html_()
        ]
        html += [
            de_genes_ref[gene]
            .head(N_GENES_TO_SHOW)
            .copy()
            .style
            .format("{:.2f}", subset=["logfoldchg", "score"])
            .format("{:.3f}", subset=["pval_adj"])
            .map(style_overlap, subset=["gene"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"REF - {gene}")
            ._repr_html_()
            for gene in v
        ]
        html.append("<br /><br />")

        # Add summary for highly expressed genes
        html.append("<b><i>Overlap of highly expressed genes:</i></b> <br /><br />")

        def style_overlap(v, propgs=""):
            top_n_query_genes = highly_expressed_query[k]["gene"][:N_GENES_TO_SHOW]
            return "color:green;" if v in top_n_query_genes else "color:red;"

        html += [
            pd.DataFrame(highly_expressed_query[k])
            .head(N_GENES_TO_SHOW)
            .style
            .format("{:.2f}", subset=["average_expression"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"QUERY - {k}")
            ._repr_html_()
        ]
        html += [
            pd.DataFrame(highly_expressed_ref[gene])
            .head(N_GENES_TO_SHOW)
            .style
            .format("{:.2f}", subset=["average_expression"])
            .map(style_overlap, subset=["gene"])
            .set_table_attributes("style='display:inline'")
            .set_caption(f"REF - {gene}")
            ._repr_html_()
            for gene in v
        ]
    else:
        html.append("<b><i>No matches found</i></b>")

    display_html("".join(html), raw=True)
    with open(join("cluster-evaluation-output", f"{k}.html"), "w") as f:
        f.write("".join(html))

    print("\n")


B_Mem : Suggestions: 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 IGHMhi_memory_B 
 0.96 
 0.61 
 
 
 
 Overlap of DE genes: 
 
 QUERY - B_Mem 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 CD79A 
 6.96 
 0.000 
 125.03 
 
 
 1 
 MS4A1 
 6.76 
 0.000 
 121.31 
 
 
 2 
 LINC01857 
 5.66 
 0.000 
 52.30 
 
 
 3 
 BANK1 
 5.56 
 0.000 
 93.43 
 
 
 4 
 LINC01781 
 5.27 
 0.000 
 38.53 
 
 
 5 
 HLA-DRA 
 5.27 
 0.000 
 121.37 
 
 
 6 
 RALGPS2 
 5.20 
 0.000 
 88.75 
 
 
 7 
 FCRL2 
 5.02 
 0.000 
 34.01 
 
 
 8 
 CD24 
 4.97 
 0.000 
 53.08 
 
 
 9 
 IGHG2 
 4.90 
 0.000 
 38.80 
 
 
 10 
 HLA-DQA1 
 4.76 
 0.000 
 103.46 
 
 
 11 
 LINC00926 
 4.68 
 0.000 
 76.11 
 
 
 12 
 FCRLA 
 4.48 
 0.000 
 56.33 
 
 
 13 
 COBLL1 
 4.48 
 0.000 
 38.74 
 
 
 14 
 BLK 
 4.39 
 0.000 
 50.35 
 
 
 

 
 REF - IGHMhi_memory_B 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 MS4A1 
 7.99 
 0.000 
 179.47 
 
 
 1 
 CD79A 
 7.95 
 0.000 
 180.54 
 
 
 2 
 ALPL 
 7.32 
 0.000 
 8.86 
 
 
 3 
 BANK1 
 7.19 
 0.000 
 176.25 
 
 
 4 
 LINC01857 
 6.93 
 0.000 
 106.16 
 
 
 5 
 RALGPS2 
 6.38 
 0.000 
 169.81 
 
 
 6 
 FCRLA 
 6.33 
 0.000 
 143.88 
 
 
 7 
 IGHM 
 6.32 
 0.000 
 141.07 
 
 
 8 
 FCRL2 
 6.23 
 0.000 
 102.08 
 
 
 9 
 CD24 
 6.23 
 0.000 
 123.89 
 
 
 10 
 LINC00926 
 6.12 
 0.000 
 149.89 
 
 
 11 
 HLA-DRA 
 6.06 
 0.000 
 171.98 
 
 
 12 
 IGHG2 
 6.03 
 0.000 
 102.12 
 
 
 13 
 SPIB 
 5.96 
 0.000 
 132.65 
 
 
 14 
 COBLL1 
 5.83 
 0.000 
 117.37 
 
 
 
 Overlap of highly expressed genes: 
 
 QUERY - B_Mem 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 MALAT1 
 7.29 
 
 
 1 
 CD74 
 7.00 
 
 
 2 
 RPS12 
 6.62 
 
 
 3 
 HLA-DRA 
 5.70 
 
 
 4 
 RPL41 
 5.55 
 
 
 5 
 ACTB 
 5.45 
 
 
 6 
 FTL 
 5.18 
 
 
 7 
 FTH1 
 5.01 
 
 
 8 
 HLA-DRB1 
 4.80 
 
 
 9 
 MT-ATP8 
 4.67 
 
 
 10 
 HLA-DPA1 
 4.65 
 
 
 11 
 TSC22D3 
 4.46 
 
 
 12 
 HLA-DPB1 
 4.35 
 
 
 13 
 CD79A 
 4.35 
 
 
 14 
 RPS26 
 4.20 
 
 
 

 
 REF - IGHMhi_memory_B 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 MALAT1 
 7.22 
 
 
 1 
 CD74 
 7.00 
 
 
 2 
 RPL41 
 6.33 
 
 
 3 
 RPS12 
 6.21 
 
 
 4 
 HLA-DRA 
 5.60 
 
 
 5 
 ACTB 
 5.31 
 
 
 6 
 FTL 
 4.88 
 
 
 7 
 FTH1 
 4.88 
 
 
 8 
 HLA-DRB1 
 4.85 
 
 
 9 
 JUNB 
 4.78 
 
 
 10 
 HLA-DPB1 
 4.72 
 
 
 11 
 HLA-DPA1 
 4.71 
 
 
 12 
 CD79A 
 4.68 
 
 
 13 
 LTB 
 4.56 
 
 
 14 
 MS4A1 
 4.54

B_Mem_Prolif : Suggestions: 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 IGHMlo_memory_B 
 1.00 
 0.50 
 
 
 
 Overlap of DE genes: 
 
 QUERY - B_Mem_Prolif 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 COCH 
 7.67 
 0.000 
 45.26 
 
 
 1 
 MS4A1 
 7.03 
 0.000 
 130.59 
 
 
 2 
 LINC01781 
 6.91 
 0.000 
 69.57 
 
 
 3 
 CD79A 
 6.88 
 0.000 
 128.36 
 
 
 4 
 IGHE 
 6.25 
 0.000 
 14.27 
 
 
 5 
 BANK1 
 6.09 
 0.000 
 110.88 
 
 
 6 
 BLK 
 5.87 
 0.000 
 85.72 
 
 
 7 
 CPNE5 
 5.71 
 0.000 
 61.43 
 
 
 8 
 HLA-DQA1 
 5.50 
 0.000 
 122.84 
 
 
 9 
 HLA-DRA 
 5.48 
 0.000 
 129.13 
 
 
 10 
 POU2AF1 
 5.39 
 0.000 
 75.04 
 
 
 11 
 RASSF6 
 5.37 
 0.000 
 10.64 
 
 
 12 
 IGHA2 
 5.06 
 0.000 
 30.47 
 
 
 13 
 SPIB 
 5.04 
 0.000 
 62.57 
 
 
 14 
 CD24 
 4.98 
 0.000 
 56.98 
 
 
 

 
 REF - IGHMlo_memory_B 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 COCH 
 8.17 
 0.000 
 95.69 
 
 
 1 
 MS4A1 
 7.91 
 0.000 
 214.56 
 
 
 2 
 LINC01781 
 7.71 
 0.000 
 117.39 
 
 
 3 
 CD79A 
 7.68 
 0.000 
 213.08 
 
 
 4 
 BANK1 
 7.25 
 0.000 
 211.76 
 
 
 5 
 POU2AF1 
 6.61 
 0.000 
 175.11 
 
 
 6 
 BLK 
 6.56 
 0.000 
 177.96 
 
 
 7 
 IGHE 
 6.40 
 0.000 
 17.99 
 
 
 8 
 LINC00926 
 6.29 
 0.000 
 182.90 
 
 
 9 
 HLA-DRA 
 6.26 
 0.000 
 212.36 
 
 
 10 
 HLA-DQA1 
 5.92 
 0.000 
 207.50 
 
 
 11 
 RASSF6 
 5.90 
 0.000 
 35.06 
 
 
 12 
 CPNE5 
 5.87 
 0.000 
 127.35 
 
 
 13 
 CD24 
 5.86 
 0.000 
 131.07 
 
 
 14 
 IGHG2 
 5.82 
 0.000 
 108.48 
 
 
 
 Overlap of highly expressed genes: 
 
 QUERY - B_Mem_Prolif 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 MALAT1 
 7.16 
 
 
 1 
 CD74 
 7.07 
 
 
 2 
 RPS12 
 6.54 
 
 
 3 
 ACTB 
 5.93 
 
 
 4 
 HLA-DRA 
 5.84 
 
 
 5 
 RPL41 
 5.50 
 
 
 6 
 FTL 
 5.13 
 
 
 7 
 HLA-DRB1 
 4.95 
 
 
 8 
 FTH1 
 4.86 
 
 
 9 
 HLA-DPA1 
 4.76 
 
 
 10 
 HLA-DPB1 
 4.60 
 
 
 11 
 MT-ATP8 
 4.58 
 
 
 12 
 LTB 
 4.55 
 
 
 13 
 TSC22D3 
 4.42 
 
 
 14 
 RPS26 
 4.31 
 
 
 

 
 REF - IGHMlo_memory_B 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 MALAT1 
 7.17 
 
 
 1 
 CD74 
 7.09 
 
 
 2 
 RPL41 
 6.39 
 
 
 3 
 RPS12 
 6.34 
 
 
 4 
 HLA-DRA 
 5.71 
 
 
 5 
 ACTB 
 5.57 
 
 
 6 
 FTL 
 4.93 
 
 
 7 
 JUN 
 4.82 
 
 
 8 
 HLA-DRB1 
 4.81 
 
 
 9 
 HLA-DPB1 
 4.79 
 
 
 10 
 LTB 
 4.73 
 
 
 11 
 HLA-DPA1 
 4.69 
 
 
 12 
 FTH1 
 4.69 
 
 
 13 
 MT-ATP8 
 4.47 
 
 
 14 
 JUNB 
 4.45

B_Naive : Suggestions: 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 naive_B 
 1.00 
 0.51 
 
 
 
 Overlap of DE genes: 
 
 QUERY - B_Naive 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 TCL1A 
 9.29 
 0.000 
 238.55 
 
 
 1 
 RP11-164H13.1 
 8.25 
 0.000 
 34.48 
 
 
 2 
 CD79A 
 8.09 
 0.000 
 307.94 
 
 
 3 
 RP11-265P11.1 
 7.92 
 0.000 
 8.20 
 
 
 4 
 MS4A1 
 7.79 
 0.000 
 293.65 
 
 
 5 
 FCER2 
 7.45 
 0.000 
 191.78 
 
 
 6 
 TCL1B 
 7.37 
 0.000 
 8.91 
 
 
 7 
 KCNG1 
 7.22 
 0.000 
 10.53 
 
 
 8 
 LINC00926 
 7.20 
 0.000 
 233.88 
 
 
 9 
 IGHM 
 7.15 
 0.000 
 250.28 
 
 
 10 
 COL19A1 
 6.93 
 0.000 
 72.08 
 
 
 11 
 IGHD 
 6.93 
 0.000 
 132.65 
 
 
 12 
 SLC38A11 
 6.89 
 0.000 
 14.30 
 
 
 13 
 DSP 
 6.57 
 0.000 
 8.54 
 
 
 14 
 VPREB3 
 6.52 
 0.000 
 147.20 
 
 
 

 
 REF - naive_B 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 TCL1A 
 10.75 
 0.000 
 279.88 
 
 
 1 
 RP11-164H13.1 
 9.00 
 0.000 
 66.04 
 
 
 2 
 RP11-265P11.1 
 8.87 
 0.000 
 30.20 
 
 
 3 
 FCER2 
 8.51 
 0.000 
 268.02 
 
 
 4 
 CD79A 
 8.48 
 0.000 
 296.88 
 
 
 5 
 MS4A1 
 8.47 
 0.000 
 293.43 
 
 
 6 
 IGHM 
 8.27 
 0.000 
 287.52 
 
 
 7 
 KCNG1 
 8.16 
 0.000 
 63.24 
 
 
 8 
 IGHD 
 8.12 
 0.000 
 221.99 
 
 
 9 
 TCL1B 
 7.89 
 0.000 
 10.29 
 
 
 10 
 LINC00926 
 7.52 
 0.000 
 266.17 
 
 
 11 
 FCRL1 
 7.48 
 0.000 
 239.35 
 
 
 12 
 BMP3 
 7.34 
 0.000 
 4.57 
 
 
 13 
 COL19A1 
 7.25 
 0.000 
 154.59 
 
 
 14 
 DSP 
 7.23 
 0.000 
 43.91 
 
 
 
 Overlap of highly expressed genes: 
 
 QUERY - B_Naive 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 7.15 
 
 
 1 
 MALAT1 
 7.07 
 
 
 2 
 RPS12 
 6.38 
 
 
 3 
 HLA-DRA 
 5.83 
 
 
 4 
 ACTB 
 5.24 
 
 
 5 
 RPL41 
 5.18 
 
 
 6 
 HLA-DRB1 
 5.12 
 
 
 7 
 FTL 
 5.00 
 
 
 8 
 FTH1 
 4.93 
 
 
 9 
 HLA-DPA1 
 4.65 
 
 
 10 
 TSC22D3 
 4.45 
 
 
 11 
 MT-ATP8 
 4.43 
 
 
 12 
 HLA-DPB1 
 4.39 
 
 
 13 
 CD79A 
 4.28 
 
 
 14 
 DUSP1 
 4.15 
 
 
 

 
 REF - naive_B 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 7.18 
 
 
 1 
 MALAT1 
 6.98 
 
 
 2 
 RPL41 
 6.17 
 
 
 3 
 RPS12 
 6.08 
 
 
 4 
 HLA-DRA 
 5.72 
 
 
 5 
 ACTB 
 5.41 
 
 
 6 
 HLA-DRB1 
 5.07 
 
 
 7 
 HLA-DPB1 
 4.80 
 
 
 8 
 FTH1 
 4.79 
 
 
 9 
 HLA-DPA1 
 4.75 
 
 
 10 
 FTL 
 4.73 
 
 
 11 
 CD79A 
 4.73 
 
 
 12 
 JUN 
 4.68 
 
 
 13 
 JUNB 
 4.56 
 
 
 14 
 MS4A1 
 4.52

B_Preplasma : Suggestions: 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 atypical_B 
 0.98 
 0.68 
 
 
 
 Overlap of DE genes: 
 
 QUERY - B_Preplasma 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 MS4A1 
 7.80 
 0.000 
 109.75 
 
 
 1 
 CD79A 
 7.61 
 0.000 
 108.32 
 
 
 2 
 FCRL5 
 6.46 
 0.000 
 54.10 
 
 
 3 
 HLA-DQA1 
 5.98 
 0.000 
 103.31 
 
 
 4 
 CD19 
 5.92 
 0.000 
 74.16 
 
 
 5 
 PPP1R14A 
 5.68 
 0.000 
 52.45 
 
 
 6 
 BANK1 
 5.65 
 0.000 
 82.42 
 
 
 7 
 HLA-DRA 
 5.64 
 0.000 
 104.29 
 
 
 8 
 FCRLA 
 5.58 
 0.000 
 70.29 
 
 
 9 
 FCRL2 
 5.51 
 0.000 
 39.15 
 
 
 10 
 IGHV1-2 
 5.47 
 0.000 
 24.77 
 
 
 11 
 MACROD2 
 5.45 
 0.000 
 21.44 
 
 
 12 
 IGHV3-43 
 5.24 
 0.000 
 20.64 
 
 
 13 
 IGHG3 
 5.17 
 0.000 
 35.06 
 
 
 14 
 HLA-DQB1 
 5.00 
 0.000 
 95.61 
 
 
 

 
 REF - atypical_B 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 MFAP5 
 9.28 
 0.000 
 4.56 
 
 
 1 
 MS4A1 
 8.49 
 0.000 
 104.91 
 
 
 2 
 CD79A 
 8.00 
 0.000 
 102.72 
 
 
 3 
 FCRL5 
 7.42 
 0.000 
 71.33 
 
 
 4 
 BANK1 
 6.84 
 0.000 
 97.89 
 
 
 5 
 FCRLA 
 6.78 
 0.000 
 92.20 
 
 
 6 
 CD19 
 6.76 
 0.000 
 90.62 
 
 
 7 
 IGHG3 
 6.47 
 0.000 
 55.31 
 
 
 8 
 FGF2 
 6.38 
 0.005 
 3.10 
 
 
 9 
 HLA-DRA 
 6.35 
 0.000 
 101.14 
 
 
 10 
 IGHG2 
 6.25 
 0.000 
 65.59 
 
 
 11 
 FCRL2 
 6.14 
 0.000 
 63.66 
 
 
 12 
 HLA-DQA1 
 6.13 
 0.000 
 99.94 
 
 
 13 
 IGHG1 
 6.07 
 0.000 
 59.36 
 
 
 14 
 PPP1R14A 
 5.86 
 0.000 
 52.33 
 
 
 
 Overlap of highly expressed genes: 
 
 QUERY - B_Preplasma 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 7.21 
 
 
 1 
 MALAT1 
 7.17 
 
 
 2 
 RPS12 
 6.21 
 
 
 3 
 HLA-DRA 
 5.98 
 
 
 4 
 ACTB 
 5.62 
 
 
 5 
 HLA-DRB1 
 5.45 
 
 
 6 
 HLA-DPA1 
 5.15 
 
 
 7 
 RPL41 
 5.12 
 
 
 8 
 FTH1 
 5.06 
 
 
 9 
 HLA-DPB1 
 4.95 
 
 
 10 
 FTL 
 4.87 
 
 
 11 
 CD79A 
 4.82 
 
 
 12 
 MS4A1 
 4.81 
 
 
 13 
 HLA-DQA1 
 4.55 
 
 
 14 
 MT-ATP8 
 4.49 
 
 
 

 
 REF - atypical_B 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 7.29 
 
 
 1 
 MALAT1 
 7.21 
 
 
 2 
 RPL41 
 6.07 
 
 
 3 
 RPS12 
 5.86 
 
 
 4 
 HLA-DRA 
 5.84 
 
 
 5 
 ACTB 
 5.42 
 
 
 6 
 HLA-DRB1 
 5.23 
 
 
 7 
 HLA-DPB1 
 5.07 
 
 
 8 
 HLA-DPA1 
 5.03 
 
 
 9 
 MS4A1 
 5.00 
 
 
 10 
 FTH1 
 4.88 
 
 
 11 
 CD79A 
 4.82 
 
 
 12 
 FTL 
 4.73 
 
 
 13 
 JUNB 
 4.65 
 
 
 14 
 HLA-DQA1 
 4.47

NKT : Suggestions: 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 NK 
 0.51 
 0.96 
 
 
 CD8+_T_GZMB+ 
 0.40 
 0.75 
 
 
 
 Overlap of DE genes: 
 
 QUERY - NKT 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 GNLY 
 7.55 
 0.000 
 204.20 
 
 
 1 
 NKG7 
 6.73 
 0.000 
 210.43 
 
 
 2 
 GZMB 
 6.22 
 0.000 
 201.73 
 
 
 3 
 CCL5 
 6.11 
 0.000 
 198.40 
 
 
 4 
 GZMH 
 5.93 
 0.000 
 194.16 
 
 
 5 
 FGFBP2 
 5.93 
 0.000 
 191.69 
 
 
 6 
 CST7 
 5.83 
 0.000 
 207.91 
 
 
 7 
 PRF1 
 5.76 
 0.000 
 195.31 
 
 
 8 
 KLRC2 
 5.57 
 0.000 
 87.41 
 
 
 9 
 GZMA 
 5.33 
 0.000 
 181.89 
 
 
 10 
 TKTL1 
 5.30 
 0.000 
 49.65 
 
 
 11 
 KLRD1 
 5.28 
 0.000 
 180.65 
 
 
 12 
 CTSW 
 5.15 
 0.000 
 190.98 
 
 
 13 
 KLRC3 
 5.13 
 0.000 
 100.31 
 
 
 14 
 KLRF1 
 5.04 
 0.000 
 143.46 
 
 
 

 
 REF - NK 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 PPBP 
 6.64 
 0.000 
 122.72 
 
 
 1 
 GNLY 
 6.63 
 0.000 
 105.74 
 
 
 2 
 NKG7 
 6.06 
 0.000 
 105.01 
 
 
 3 
 GZMB 
 5.39 
 0.000 
 104.25 
 
 
 4 
 PF4 
 5.28 
 0.000 
 82.58 
 
 
 5 
 GP1BB 
 5.24 
 0.000 
 76.27 
 
 
 6 
 TUBB1 
 5.14 
 0.000 
 72.35 
 
 
 7 
 FGFBP2 
 4.92 
 0.000 
 97.18 
 
 
 8 
 CAVIN2 
 4.85 
 0.000 
 70.27 
 
 
 9 
 PRF1 
 4.76 
 0.000 
 100.05 
 
 
 10 
 CST7 
 4.76 
 0.000 
 98.93 
 
 
 11 
 NRGN 
 4.75 
 0.000 
 92.50 
 
 
 12 
 SPARC 
 4.68 
 0.000 
 60.89 
 
 
 13 
 CCL5 
 4.67 
 0.000 
 87.27 
 
 
 14 
 MYL9 
 4.35 
 0.000 
 46.08 
 
 
 

 
 REF - CD8+_T_GZMB+ 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 NKG7 
 6.40 
 0.000 
 330.52 
 
 
 1 
 GZMH 
 6.20 
 0.000 
 361.85 
 
 
 2 
 CCL5 
 5.90 
 0.000 
 341.57 
 
 
 3 
 GNLY 
 5.17 
 0.000 
 245.52 
 
 
 4 
 CD8A 
 5.13 
 0.000 
 307.58 
 
 
 5 
 CST7 
 5.09 
 0.000 
 316.50 
 
 
 6 
 GZMB 
 4.78 
 0.000 
 277.08 
 
 
 7 
 FGFBP2 
 4.74 
 0.000 
 277.57 
 
 
 8 
 GZMA 
 4.56 
 0.000 
 287.98 
 
 
 9 
 ZNF683 
 4.44 
 0.000 
 122.20 
 
 
 10 
 PRF1 
 4.44 
 0.000 
 281.33 
 
 
 11 
 TRGV2 
 4.43 
 0.000 
 105.12 
 
 
 12 
 TRGV4 
 4.38 
 0.000 
 80.98 
 
 
 13 
 KLRD1 
 4.25 
 0.000 
 268.78 
 
 
 14 
 TRDV3 
 4.20 
 0.000 
 15.02 
 
 
 
 Overlap of highly expressed genes: 
 
 QUERY - NKT 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 MALAT1 
 7.14 
 
 
 1 
 ACTB 
 6.47 
 
 
 2 
 NKG7 
 6.24 
 
 
 3 
 GNLY 
 6.08 
 
 
 4 
 RPS12 
 5.88 
 
 
 5 
 CCL5 
 5.64 
 
 
 6 
 S100A4 
 5.27 
 
 
 7 
 IFITM1 
 5.16 
 
 
 8 
 FTL 
 4.91 
 
 
 9 
 RPL41 
 4.90 
 
 
 10 
 CST7 
 4.89 
 
 
 11 
 IFITM2 
 4.63 
 
 
 12 
 IL32 
 4.54 
 
 
 13 
 FTH1 
 4.53 
 
 
 14 
 GZMA 
 4.47 
 
 
 

 
 REF - NK 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 MALAT1 
 7.07 
 
 
 1 
 ACTB 
 6.15 
 
 
 2 
 NKG7 
 5.92 
 
 
 3 
 GNLY 
 5.80 
 
 
 4 
 RPL41 
 5.65 
 
 
 5 
 RPS12 
 5.49 
 
 
 6 
 FTH1 
 4.97 
 
 
 7 
 CCL5 
 4.90 
 
 
 8 
 IFITM1 
 4.78 
 
 
 9 
 DUSP1 
 4.66 
 
 
 10 
 FOS 
 4.66 
 
 
 11 
 FTL 
 4.61 
 
 
 12 
 CST7 
 4.44 
 
 
 13 
 JUNB 
 4.41 
 
 
 14 
 IFITM2 
 4.35 
 
 
 

 
 REF - CD8+_T_GZMB+ 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 MALAT1 
 7.37 
 
 
 1 
 ACTB 
 6.21 
 
 
 2 
 RPL41 
 6.07 
 
 
 3 
 RPS12 
 6.03 
 
 
 4 
 NKG7 
 5.88 
 
 
 5 
 CCL5 
 5.49 
 
 
 6 
 IL32 
 4.99 
 
 
 7 
 S100A4 
 4.94 
 
 
 8 
 JUN 
 4.68 
 
 
 9 
 IFITM1 
 4.67 
 
 
 10 
 FTL 
 4.65 
 
 
 11 
 GNLY 
 4.57 
 
 
 12 
 FTH1 
 4.51 
 
 
 13 
 JUNB 
 4.50 
 
 
 14 
 CST7 
 4.43

...

cDC_1 : Suggestions: 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 cDC1 
 1.00 
 0.33 
 
 
 
 Overlap of DE genes: 
 
 QUERY - cDC_1 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 CLEC9A 
 12.84 
 0.000 
 36.07 
 
 
 1 
 IDO1 
 10.41 
 0.000 
 25.94 
 
 
 2 
 DNASE1L3 
 8.25 
 0.000 
 23.10 
 
 
 3 
 CST3 
 6.50 
 0.000 
 37.68 
 
 
 4 
 HLA-DQA1 
 6.43 
 0.000 
 36.09 
 
 
 5 
 CPVL 
 6.20 
 0.000 
 36.50 
 
 
 6 
 SERPINF1 
 6.19 
 0.000 
 25.48 
 
 
 7 
 HLA-DRA 
 6.17 
 0.000 
 37.24 
 
 
 8 
 EGLN3 
 6.12 
 0.000 
 12.77 
 
 
 9 
 IRF8 
 6.08 
 0.000 
 34.94 
 
 
 10 
 HLA-DPA1 
 6.06 
 0.000 
 38.44 
 
 
 11 
 HLA-DQB1 
 5.97 
 0.000 
 36.15 
 
 
 12 
 HLA-DPB1 
 5.92 
 0.000 
 38.21 
 
 
 13 
 APOC1 
 5.69 
 0.000 
 5.01 
 
 
 14 
 HLA-DRB1 
 5.50 
 0.000 
 37.48 
 
 
 

 
 REF - cDC1 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 CLEC9A 
 13.55 
 0.000 
 44.64 
 
 
 1 
 IDO1 
 11.72 
 0.000 
 39.24 
 
 
 2 
 DNASE1L3 
 9.14 
 0.000 
 37.64 
 
 
 3 
 CST3 
 7.48 
 0.000 
 44.72 
 
 
 4 
 CPVL 
 7.10 
 0.000 
 44.56 
 
 
 5 
 KCND3 
 7.03 
 0.000 
 7.41 
 
 
 6 
 HLA-DRA 
 6.96 
 0.000 
 44.48 
 
 
 7 
 HLA-DQA1 
 6.91 
 0.000 
 43.89 
 
 
 8 
 APOC1 
 6.76 
 0.000 
 7.78 
 
 
 9 
 IRF8 
 6.60 
 0.000 
 44.01 
 
 
 10 
 HLA-DQB1 
 6.32 
 0.000 
 43.86 
 
 
 11 
 EGLN3 
 6.31 
 0.000 
 28.27 
 
 
 12 
 HLA-DPA1 
 6.26 
 0.000 
 44.94 
 
 
 13 
 HLA-DRB1 
 6.15 
 0.000 
 44.29 
 
 
 14 
 SERPINF1 
 6.09 
 0.000 
 35.89 
 
 
 
 Overlap of highly expressed genes: 
 
 QUERY - cDC_1 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 7.45 
 
 
 1 
 ACTB 
 6.90 
 
 
 2 
 HLA-DRA 
 6.37 
 
 
 3 
 CST3 
 6.33 
 
 
 4 
 HLA-DPA1 
 6.07 
 
 
 5 
 HLA-DRB1 
 6.02 
 
 
 6 
 MALAT1 
 5.91 
 
 
 7 
 HLA-DPB1 
 5.80 
 
 
 8 
 RPS12 
 5.68 
 
 
 9 
 FTH1 
 5.52 
 
 
 10 
 VIM 
 5.31 
 
 
 11 
 LYZ 
 5.27 
 
 
 12 
 S100A10 
 4.98 
 
 
 13 
 HLA-DQA1 
 4.89 
 
 
 14 
 HLA-DQB1 
 4.76 
 
 
 

 
 REF - cDC1 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 7.53 
 
 
 1 
 ACTB 
 6.65 
 
 
 2 
 CST3 
 6.32 
 
 
 3 
 HLA-DRA 
 6.28 
 
 
 4 
 HLA-DPA1 
 5.99 
 
 
 5 
 HLA-DRB1 
 5.92 
 
 
 6 
 HLA-DPB1 
 5.89 
 
 
 7 
 MALAT1 
 5.83 
 
 
 8 
 VIM 
 5.51 
 
 
 9 
 RPL41 
 5.44 
 
 
 10 
 FTH1 
 5.42 
 
 
 11 
 RPS12 
 5.38 
 
 
 12 
 LYZ 
 5.27 
 
 
 13 
 FOS 
 5.14 
 
 
 14 
 HLA-DRB5 
 5.05

cDC_2 : Suggestions: 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 cDC2 
 0.64 
 0.61 
 
 
 cDC 
 0.49 
 0.63 
 
 
 
 Overlap of DE genes: 
 
 QUERY - cDC_2 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 FCER1A 
 9.78 
 0.000 
 116.28 
 
 
 1 
 CLEC10A 
 7.40 
 0.000 
 113.18 
 
 
 2 
 CD1C 
 7.23 
 0.000 
 101.92 
 
 
 3 
 CST3 
 5.83 
 0.000 
 137.70 
 
 
 4 
 HLA-DRA 
 5.82 
 0.000 
 134.30 
 
 
 5 
 HLA-DQA1 
 5.61 
 0.000 
 123.47 
 
 
 6 
 HLA-DQB1 
 5.25 
 0.000 
 126.20 
 
 
 7 
 PLD4 
 5.21 
 0.000 
 89.89 
 
 
 8 
 HLA-DRB1 
 5.06 
 0.000 
 136.52 
 
 
 9 
 HLA-DPB1 
 4.92 
 0.000 
 130.63 
 
 
 10 
 HLA-DPA1 
 4.91 
 0.000 
 131.59 
 
 
 11 
 APOC1 
 4.89 
 0.000 
 8.08 
 
 
 12 
 LYZ 
 4.88 
 0.000 
 103.24 
 
 
 13 
 CPVL 
 4.75 
 0.000 
 120.91 
 
 
 14 
 HLA-DMA 
 4.45 
 0.000 
 121.02 
 
 
 

 
 REF - cDC2 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 FCER1A 
 9.68 
 0.000 
 151.32 
 
 
 1 
 CLEC10A 
 7.95 
 0.000 
 160.58 
 
 
 2 
 APOC1 
 6.74 
 0.000 
 13.21 
 
 
 3 
 CD1C 
 6.66 
 0.000 
 138.75 
 
 
 4 
 CST3 
 6.61 
 0.000 
 191.34 
 
 
 5 
 IL1R2 
 6.31 
 0.000 
 51.28 
 
 
 6 
 HLA-DRA 
 6.30 
 0.000 
 186.12 
 
 
 7 
 LYZ 
 6.24 
 0.000 
 158.51 
 
 
 8 
 HLA-DQA1 
 5.83 
 0.000 
 177.06 
 
 
 9 
 HLA-DRB1 
 5.42 
 0.000 
 186.61 
 
 
 10 
 HLA-DQB1 
 5.33 
 0.000 
 177.28 
 
 
 11 
 CPVL 
 5.19 
 0.000 
 176.47 
 
 
 12 
 IFI30 
 5.15 
 0.000 
 149.48 
 
 
 13 
 HLA-DRB5 
 4.99 
 0.000 
 170.29 
 
 
 14 
 LGALS2 
 4.83 
 0.000 
 158.37 
 
 
 

 
 REF - cDC 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 FCER1A 
 7.90 
 0.000 
 21.73 
 
 
 1 
 CLEC9A 
 7.13 
 0.000 
 4.18 
 
 
 2 
 CST3 
 6.79 
 0.000 
 28.01 
 
 
 3 
 CLEC10A 
 6.71 
 0.000 
 20.45 
 
 
 4 
 HLA-DRA 
 6.70 
 0.000 
 28.05 
 
 
 5 
 CD1C 
 6.61 
 0.000 
 21.27 
 
 
 6 
 IDO1 
 6.52 
 0.000 
 3.86 
 
 
 7 
 HLA-DQA1 
 6.52 
 0.000 
 27.56 
 
 
 8 
 APOC1 
 6.24 
 0.006 
 3.15 
 
 
 9 
 HLA-DQB1 
 5.90 
 0.000 
 27.40 
 
 
 10 
 IL1R2 
 5.81 
 0.000 
 8.53 
 
 
 11 
 LYZ 
 5.76 
 0.000 
 21.08 
 
 
 12 
 HLA-DRB1 
 5.73 
 0.000 
 27.83 
 
 
 13 
 PPBP 
 5.49 
 0.000 
 25.46 
 
 
 14 
 HLA-DPA1 
 5.43 
 0.000 
 27.99 
 
 
 
 Overlap of highly expressed genes: 
 
 QUERY - cDC_2 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 6.87 
 
 
 1 
 ACTB 
 6.75 
 
 
 2 
 HLA-DRA 
 6.07 
 
 
 3 
 MALAT1 
 5.93 
 
 
 4 
 CST3 
 5.82 
 
 
 5 
 FTH1 
 5.71 
 
 
 6 
 HLA-DRB1 
 5.67 
 
 
 7 
 LYZ 
 5.60 
 
 
 8 
 RPS12 
 5.56 
 
 
 9 
 FTL 
 5.41 
 
 
 10 
 S100A4 
 5.25 
 
 
 11 
 HLA-DPA1 
 5.23 
 
 
 12 
 VIM 
 5.15 
 
 
 13 
 HLA-DPB1 
 5.06 
 
 
 14 
 S100A6 
 5.01 
 
 
 

 
 REF - cDC2 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 6.76 
 
 
 1 
 ACTB 
 6.29 
 
 
 2 
 MALAT1 
 6.02 
 
 
 3 
 FTH1 
 5.80 
 
 
 4 
 HLA-DRA 
 5.76 
 
 
 5 
 CST3 
 5.65 
 
 
 6 
 RPL41 
 5.56 
 
 
 7 
 LYZ 
 5.50 
 
 
 8 
 FOS 
 5.46 
 
 
 9 
 VIM 
 5.40 
 
 
 10 
 RPS12 
 5.39 
 
 
 11 
 HLA-DRB1 
 5.36 
 
 
 12 
 FTL 
 5.34 
 
 
 13 
 HLA-DPB1 
 5.00 
 
 
 14 
 HLA-DPA1 
 4.95 
 
 
 

 
 REF - cDC 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 7.07 
 
 
 1 
 ACTB 
 6.39 
 
 
 2 
 HLA-DRA 
 6.10 
 
 
 3 
 FTH1 
 5.84 
 
 
 4 
 CST3 
 5.84 
 
 
 5 
 MALAT1 
 5.83 
 
 
 6 
 HLA-DRB1 
 5.63 
 
 
 7 
 RPL41 
 5.51 
 
 
 8 
 HLA-DPB1 
 5.47 
 
 
 9 
 HLA-DPA1 
 5.42 
 
 
 10 
 RPS12 
 5.38 
 
 
 11 
 VIM 
 5.35 
 
 
 12 
 FOS 
 5.32 
 
 
 13 
 LYZ 
 5.24 
 
 
 14 
 FTL 
 4.94

cM : Suggestions: 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 CD14+_Monocyte 
 0.99 
 0.63 
 
 
 
 Overlap of DE genes: 
 
 QUERY - cM 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 S100A12 
 7.25 
 0.000 
 517.73 
 
 
 1 
 CD14 
 7.21 
 0.000 
 511.92 
 
 
 2 
 S100A8 
 7.13 
 0.000 
 577.76 
 
 
 3 
 LYZ 
 7.11 
 0.000 
 574.97 
 
 
 4 
 S100A9 
 6.91 
 0.000 
 576.15 
 
 
 5 
 VCAN 
 6.87 
 0.000 
 524.40 
 
 
 6 
 FCN1 
 6.81 
 0.000 
 539.78 
 
 
 7 
 CST3 
 6.45 
 0.000 
 518.12 
 
 
 8 
 MS4A6A 
 6.09 
 0.000 
 468.99 
 
 
 9 
 HP 
 6.06 
 0.000 
 73.87 
 
 
 10 
 CSTA 
 6.01 
 0.000 
 474.75 
 
 
 11 
 MCEMP1 
 5.98 
 0.000 
 183.74 
 
 
 12 
 FOLR3 
 5.95 
 0.000 
 113.69 
 
 
 13 
 CLEC4E 
 5.87 
 0.000 
 202.11 
 
 
 14 
 MNDA 
 5.77 
 0.000 
 485.17 
 
 
 

 
 REF - CD14+_Monocyte 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 S100A8 
 9.37 
 0.000 
 651.10 
 
 
 1 
 S100A9 
 8.87 
 0.000 
 651.36 
 
 
 2 
 LYZ 
 8.35 
 0.000 
 646.92 
 
 
 3 
 S100A12 
 7.88 
 0.000 
 568.19 
 
 
 4 
 VCAN 
 7.69 
 0.000 
 611.23 
 
 
 5 
 FCN1 
 7.25 
 0.000 
 620.43 
 
 
 6 
 CD14 
 7.25 
 0.000 
 574.61 
 
 
 7 
 CST3 
 6.88 
 0.000 
 590.38 
 
 
 8 
 CSF3R 
 6.73 
 0.000 
 561.72 
 
 
 9 
 IFI30 
 6.65 
 0.000 
 591.96 
 
 
 10 
 MS4A6A 
 6.45 
 0.000 
 547.80 
 
 
 11 
 MNDA 
 6.36 
 0.000 
 573.68 
 
 
 12 
 TNFAIP2 
 6.31 
 0.000 
 567.47 
 
 
 13 
 CLEC4E 
 6.15 
 0.000 
 265.19 
 
 
 14 
 RP11-1143G9.4 
 6.15 
 0.000 
 454.63 
 
 
 
 Overlap of highly expressed genes: 
 
 QUERY - cM 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 S100A8 
 6.57 
 
 
 1 
 S100A9 
 6.57 
 
 
 2 
 FTL 
 6.38 
 
 
 3 
 ACTB 
 6.13 
 
 
 4 
 MALAT1 
 5.90 
 
 
 5 
 FTH1 
 5.77 
 
 
 6 
 LYZ 
 5.71 
 
 
 7 
 S100A4 
 5.62 
 
 
 8 
 S100A6 
 5.55 
 
 
 9 
 RPS12 
 5.02 
 
 
 10 
 CST3 
 4.90 
 
 
 11 
 TYROBP 
 4.87 
 
 
 12 
 VIM 
 4.80 
 
 
 13 
 CD74 
 4.66 
 
 
 14 
 FOS 
 4.63 
 
 
 

 
 REF - CD14+_Monocyte 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 FTL 
 6.29 
 
 
 1 
 FTH1 
 6.06 
 
 
 2 
 MALAT1 
 6.04 
 
 
 3 
 S100A9 
 5.88 
 
 
 4 
 S100A8 
 5.83 
 
 
 5 
 ACTB 
 5.79 
 
 
 6 
 LYZ 
 5.69 
 
 
 7 
 FOS 
 5.57 
 
 
 8 
 RPL41 
 5.34 
 
 
 9 
 S100A6 
 5.33 
 
 
 10 
 CD74 
 5.24 
 
 
 11 
 S100A4 
 5.22 
 
 
 12 
 RPS12 
 5.11 
 
 
 13 
 VIM 
 4.89 
 
 
 14 
 TYROBP 
 4.84

ncM : Suggestions: 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 CD16+_Monocyte 
 1.00 
 0.51 
 
 
 
 Overlap of DE genes: 
 
 QUERY - ncM 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 LYPD2 
 7.83 
 0.000 
 43.40 
 
 
 1 
 CDKN1C 
 7.53 
 0.000 
 134.58 
 
 
 2 
 VMO1 
 7.48 
 0.000 
 45.97 
 
 
 3 
 C1QB 
 6.84 
 0.000 
 54.80 
 
 
 4 
 C1QC 
 6.67 
 0.000 
 39.41 
 
 
 5 
 C1QA 
 6.65 
 0.000 
 95.38 
 
 
 6 
 CKB 
 6.36 
 0.000 
 65.26 
 
 
 7 
 MEG3 
 6.16 
 0.000 
 5.82 
 
 
 8 
 MS4A7 
 5.99 
 0.000 
 227.79 
 
 
 9 
 HES4 
 5.99 
 0.000 
 130.29 
 
 
 10 
 FCGR3A 
 5.96 
 0.000 
 221.25 
 
 
 11 
 LST1 
 5.84 
 0.000 
 250.36 
 
 
 12 
 PELATON 
 5.66 
 0.000 
 215.87 
 
 
 13 
 IFITM3 
 5.37 
 0.000 
 230.07 
 
 
 14 
 SERPINA1 
 5.33 
 0.000 
 229.73 
 
 
 

 
 REF - CD16+_Monocyte 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 LYPD2 
 8.49 
 0.000 
 113.90 
 
 
 1 
 CDKN1C 
 8.05 
 0.000 
 292.13 
 
 
 2 
 MEG3 
 7.69 
 0.000 
 12.31 
 
 
 3 
 VMO1 
 7.41 
 0.000 
 92.40 
 
 
 4 
 C1QA 
 7.04 
 0.000 
 95.99 
 
 
 5 
 C1QB 
 6.75 
 0.000 
 30.28 
 
 
 6 
 CKB 
 6.71 
 0.000 
 166.91 
 
 
 7 
 PELATON 
 6.68 
 0.000 
 320.78 
 
 
 8 
 HES4 
 6.61 
 0.000 
 258.46 
 
 
 9 
 LST1 
 6.42 
 0.000 
 340.95 
 
 
 10 
 MS4A7 
 6.31 
 0.000 
 314.95 
 
 
 11 
 SERPINA1 
 6.28 
 0.000 
 329.64 
 
 
 12 
 IFI30 
 6.11 
 0.000 
 310.72 
 
 
 13 
 CST3 
 5.94 
 0.000 
 288.67 
 
 
 14 
 CSF1R 
 5.87 
 0.000 
 302.62 
 
 
 
 Overlap of highly expressed genes: 
 
 QUERY - ncM 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 FTL 
 6.83 
 
 
 1 
 FTH1 
 6.59 
 
 
 2 
 ACTB 
 6.52 
 
 
 3 
 MALAT1 
 6.14 
 
 
 4 
 S100A4 
 5.67 
 
 
 5 
 CD74 
 5.33 
 
 
 6 
 S100A6 
 5.31 
 
 
 7 
 CST3 
 5.30 
 
 
 8 
 RPS12 
 5.24 
 
 
 9 
 TYROBP 
 5.07 
 
 
 10 
 IFITM3 
 5.01 
 
 
 11 
 FCER1G 
 4.97 
 
 
 12 
 SAT1 
 4.95 
 
 
 13 
 LST1 
 4.86 
 
 
 14 
 COTL1 
 4.86 
 
 
 

 
 REF - CD16+_Monocyte 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 FTL 
 6.63 
 
 
 1 
 FTH1 
 6.46 
 
 
 2 
 MALAT1 
 6.29 
 
 
 3 
 ACTB 
 6.27 
 
 
 4 
 CD74 
 5.38 
 
 
 5 
 RPL41 
 5.36 
 
 
 6 
 S100A4 
 5.30 
 
 
 7 
 SAT1 
 5.26 
 
 
 8 
 FOS 
 5.22 
 
 
 9 
 RPS12 
 5.14 
 
 
 10 
 PSAP 
 5.09 
 
 
 11 
 CST3 
 5.06 
 
 
 12 
 IFI30 
 5.05 
 
 
 13 
 S100A6 
 5.03 
 
 
 14 
 TYROBP 
 4.89

pDC : Suggestions: 
 
 
 
   
 OT mass 
 distance 
 
 
 
 
 pDC 
 1.00 
 0.45 
 
 
 
 Overlap of DE genes: 
 
 QUERY - pDC 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 SCT 
 13.95 
 0.000 
 56.81 
 
 
 1 
 LRRC26 
 13.46 
 0.000 
 53.40 
 
 
 2 
 SHD 
 12.41 
 0.000 
 34.56 
 
 
 3 
 CLEC4C 
 12.05 
 0.000 
 63.44 
 
 
 4 
 LILRA4 
 11.93 
 0.000 
 74.36 
 
 
 5 
 KRT5 
 11.72 
 0.000 
 12.72 
 
 
 6 
 SERPINF1 
 9.73 
 0.000 
 70.85 
 
 
 7 
 PTPRS 
 9.71 
 0.000 
 36.38 
 
 
 8 
 TPM2 
 9.63 
 0.000 
 59.49 
 
 
 9 
 PLD4 
 9.34 
 0.000 
 78.07 
 
 
 10 
 PTCRA 
 9.20 
 0.000 
 48.57 
 
 
 11 
 DNASE1L3 
 9.20 
 0.000 
 46.59 
 
 
 12 
 IL3RA 
 8.63 
 0.000 
 68.56 
 
 
 13 
 LAMP5 
 8.60 
 0.000 
 34.78 
 
 
 14 
 ITM2C 
 8.14 
 0.000 
 76.54 
 
 
 

 
 REF - pDC 
 
 
   
 gene 
 logfoldchg 
 pval_adj 
 score 
 
 
 
 
 0 
 LRRC26 
 14.01 
 0.000 
 91.22 
 
 
 1 
 SCT 
 13.61 
 0.000 
 89.52 
 
 
 2 
 SHD 
 12.98 
 0.000 
 58.25 
 
 
 3 
 LILRA4 
 12.35 
 0.000 
 101.52 
 
 
 4 
 CLEC4C 
 12.19 
 0.000 
 97.11 
 
 
 5 
 KRT5 
 11.54 
 0.000 
 31.41 
 
 
 6 
 PTPRS 
 10.48 
 0.000 
 91.59 
 
 
 7 
 SERPINF1 
 10.22 
 0.000 
 101.46 
 
 
 8 
 LAMP5 
 10.10 
 0.000 
 76.41 
 
 
 9 
 DNASE1L3 
 9.70 
 0.000 
 79.05 
 
 
 10 
 PLD4 
 9.31 
 0.000 
 102.49 
 
 
 11 
 IL3RA 
 9.09 
 0.000 
 99.93 
 
 
 12 
 DERL3 
 9.05 
 0.000 
 96.61 
 
 
 13 
 TPM2 
 8.77 
 0.000 
 89.44 
 
 
 14 
 CRYM 
 8.63 
 0.000 
 33.34 
 
 
 
 Overlap of highly expressed genes: 
 
 QUERY - pDC 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 7.06 
 
 
 1 
 MALAT1 
 6.88 
 
 
 2 
 RPS12 
 6.26 
 
 
 3 
 ACTB 
 6.07 
 
 
 4 
 FTH1 
 5.76 
 
 
 5 
 HLA-DRA 
 5.48 
 
 
 6 
 RPL41 
 5.12 
 
 
 7 
 FTL 
 4.97 
 
 
 8 
 CST3 
 4.91 
 
 
 9 
 HLA-DRB1 
 4.82 
 
 
 10 
 GZMB 
 4.76 
 
 
 11 
 JCHAIN 
 4.65 
 
 
 12 
 HLA-DPA1 
 4.59 
 
 
 13 
 VIM 
 4.48 
 
 
 14 
 TAGLN2 
 4.39 
 
 
 

 
 REF - pDC 
 
 
   
 gene 
 average_expression 
 
 
 
 
 0 
 CD74 
 7.02 
 
 
 1 
 MALAT1 
 6.93 
 
 
 2 
 RPS12 
 5.94 
 
 
 3 
 RPL41 
 5.79 
 
 
 4 
 ACTB 
 5.65 
 
 
 5 
 FTH1 
 5.55 
 
 
 6 
 HLA-DRA 
 5.27 
 
 
 7 
 GZMB 
 4.94 
 
 
 8 
 VIM 
 4.77 
 
 
 9 
 CST3 
 4.76 
 
 
 10 
 TAGLN2 
 4.73 
 
 
 11 
 FTL 
 4.72 
 
 
 12 
 HLA-DRB1 
 4.68 
 
 
 13 
 PLD4 
 4.67 
 
 
 14 
 IRF8 
 4.54